<a href="https://colab.research.google.com/github/abikooo/Adversarial-Robustness-MNIST-CIFAR10-FGSM-Defense/blob/main/MNIST_Defense.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

In [2]:
# 1. configuration & hyperparameters
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
EPOCHS = 5
LEARNING_RATE = 0.001
EPSILON = 0.2  # how strong the adversarial attack is
TEMPERATURE = 40.0 # used for defensive distillation

In [3]:
# 2. loading dataset (choose between 'mnist' or 'cifar10')
DATASET_NAME = 'MNIST'

if DATASET_NAME == 'MNIST':
    transform = transforms.Compose([transforms.ToTensor()])
    train_set = datasets.MNIST('./data', train=True, download=True, transform=transform)
    test_set = datasets.MNIST('./data', train=False, transform=transform)
    in_channels = 1
else:
    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    train_set = datasets.CIFAR10('./data', train=True, download=True, transform=transform)
    test_set = datasets.CIFAR10('./data', train=False, transform=transform)
    in_channels = 3

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

100%|██████████| 9.91M/9.91M [00:02<00:00, 4.86MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 132kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.18MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.9MB/s]


In [4]:
# 3. simple cnn model (standard for undergrad projects)
class SimpleCNN(nn.Module):
    def __init__(self, in_channels):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.fc1 = nn.Linear(9216 if DATASET_NAME == 'MNIST' else 12544, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x, temp=1.0):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = self.fc2(x)
        return x / temp # scale output for distillation if needed

In [5]:
# 4. fgsm attack implementation
def fgsm_attack(image, epsilon, data_grad):
    # get the sign of the gradients
    sign_data_grad = data_grad.sign()
    # add small perturbation to the image
    perturbed_image = image + epsilon * sign_data_grad
    # make sure pixel values are still valid
    perturbed_image = torch.clamp(perturbed_image, 0, 1)
    return perturbed_image

In [6]:
# 5. training functions
def train_standard(model, loader, optimizer):
    model.train()
    for data, target in loader:
        data, target = data.to(DEVICE), target.to(DEVICE)
        optimizer.zero_grad()
        output = model(data)
        loss = F.cross_entropy(output, target)
        loss.backward()
        optimizer.step()

def train_adversarial(model, loader, optimizer, epsilon):
    model.train()
    for data, target in loader:
        data, target = data.to(DEVICE), target.to(DEVICE)
        data.requires_grad = True

        # generate adversarial examples on the fly
        output = model(data)
        loss = F.cross_entropy(output, target)
        model.zero_grad()
        loss.backward()

        data_grad = data.grad.data
        adv_data = fgsm_attack(data, epsilon, data_grad)

        # train on the adversarial images
        optimizer.zero_grad()
        adv_output = model(adv_data)
        adv_loss = F.cross_entropy(adv_output, target)
        adv_loss.backward()
        optimizer.step()


In [7]:
# 6. evaluation function (for clean vs attacked images)
def evaluate(model, loader, epsilon):
    model.eval()
    clean_correct = 0
    adv_correct = 0

    for data, target in loader:
        data, target = data.to(DEVICE), target.to(DEVICE)
        data.requires_grad = True

        output = model(data)
        clean_pred = output.max(1, keepdim=True)[1]
        clean_correct += clean_pred.eq(target.view_as(clean_pred)).sum().item()

        # create adversarial examples
        loss = F.cross_entropy(output, target)
        model.zero_grad()
        loss.backward()
        adv_data = fgsm_attack(data, epsilon, data.grad.data)

        adv_output = model(adv_data)
        adv_pred = adv_output.max(1, keepdim=True)[1]
        adv_correct += adv_pred.eq(target.view_as(adv_pred)).sum().item()

    return clean_correct / len(loader.dataset), adv_correct / len(loader.dataset)

In [8]:
# execution flow

# a. train baseline model
print("training baseline model...")
baseline_model = SimpleCNN(in_channels).to(DEVICE)
optimizer = optim.Adam(baseline_model.parameters(), lr=LEARNING_RATE)
for epoch in range(1, EPOCHS + 1):
    train_standard(baseline_model, train_loader, optimizer)
base_clean, base_adv = evaluate(baseline_model, test_loader, EPSILON)

# b. adversarial training
print("training adversarially robust model...")
adv_model = SimpleCNN(in_channels).to(DEVICE)
optimizer = optim.Adam(adv_model.parameters(), lr=LEARNING_RATE)
for epoch in range(1, EPOCHS + 1):
    train_adversarial(adv_model, train_loader, optimizer, EPSILON)
robust_clean, robust_adv = evaluate(adv_model, test_loader, EPSILON)

training baseline model...
training adversarially robust model...


In [9]:
# c. defensive distillation
# train teacher model with normal training
print("training distilled model (student)...")
teacher = SimpleCNN(in_channels).to(DEVICE)
t_opt = optim.Adam(teacher.parameters(), lr=LEARNING_RATE)
for _ in range(EPOCHS):
    train_standard(teacher, train_loader, t_opt)

training distilled model (student)...


In [10]:
# train student model on teacher's soft labels
student = SimpleCNN(in_channels).to(DEVICE)
s_opt = optim.Adam(student.parameters(), lr=LEARNING_RATE)
teacher.eval()
for data, target in train_loader:
    data = data.to(DEVICE)
    with torch.no_grad():
        soft_labels = F.softmax(teacher(data, temp=TEMPERATURE), dim=1)
    s_opt.zero_grad()
    student_out = F.log_softmax(student(data, temp=TEMPERATURE), dim=1)
    loss = nn.KLDivLoss(reduction='batchmean')(student_out, soft_labels)
    loss.backward()
    s_opt.step()
distill_clean, distill_adv = evaluate(student, test_loader, EPSILON)

In [11]:
# print final results table
print("\n" + "="*30)
print(f"results for {DATASET_NAME} (epsilon={EPSILON})")
print(f"{'model':<15} | {'clean acc':<10} | {'adv acc':<10}")
print("-" * 40)
print(f"{'baseline':<15} | {base_clean:.2%} | {base_adv:.2%}")
print(f"{'adv. trained':<15} | {robust_clean:.2%} | {robust_adv:.2%}")
print(f"{'distilled':<15} | {distill_clean:.2%} | {distill_adv:.2%}")


results for MNIST (epsilon=0.2)
model           | clean acc  | adv acc   
----------------------------------------
baseline        | 99.11% | 62.41%
adv. trained    | 98.81% | 94.39%
distilled       | 98.04% | 27.88%
